# Unknown-prefix invariant PR codes

Recover the **message**, without knowing how many initial channel symbols were removed.
Each trial independently samples a protocol, message length, overhead, SNR and prefix length.
The experiment uses zero padding and a nonzero pilot field block; the decoder ignores recovered padding.

[Construction and OSD](../../docs/shift_invariant_pr.md) ? [Time and memory analysis](../../docs/shift_invariant_pr_methods.md)

The implementation uses sieve/Frobenius carrier enumeration and exp/log phase repair.
The [theoretical analysis](../../docs/shift_invariant_pr_methods.md) explains their time and memory tradeoffs.


In [1]:
from pathlib import Path
import sys

for parent in (Path.cwd(), *Path.cwd().parents):
    candidate = parent / "notebooks"
    if (candidate / "_shared").is_dir():
        ROOT = parent
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Cannot locate notebooks/_shared")

from IPython.display import display
import pandas as pd
from joblib import cpu_count
from _shared.core.finite_fields import choose_degree
from _shared.core.decoders.x.osd import OSDConfig
from _shared.core.protocols.shift_invariant_pr import ShiftInvariantPR
from _shared.application.protocol_experiment import (
    ExperimentConfig,
    prepare_protocols,
    warmup_protocols,
    run_experiment,
)

## Configuration

`k` is selected among the three nearest integers to `log2(C*K)`, minimizing
`(padding + k, padding, k)` lexicographically, with `padding = (-K) % k`.
The complete unknown word has `D = K + padding + k` bits, including the phase block.
**N = ceil((1 + Overhead) * D)**: zero padding costs channel observations.

SNR means **Es/N0**, with Es=1, noise variance `1 / (2 * 10**(SNR/10))`, and channel LLR
`2 * received / variance`. `S_RANGE` is inclusive. Phase recovery works modulo the field period;
S itself is never passed to the decoder.

OSD order 1 searches all single flips of the most reliable basis. `OSD_SEARCH_BITS=0`
uses the full basis; a positive value searches its least reliable subset. The candidate cap
includes the unflipped word; increasing order/cap trades runtime for decoding performance.


In [2]:
C = 32
K_VALUES = (32, 64, 128, 256)
EXPEREMENTS = 1_000_000
SNR_DB_RANGE = (-3.0, 3.0)
OVERHEAD_RANGE = (0.1, 0.5)
S_RANGE = (0, 4095)
BATCH_SIZE = 256
WORKERS = min(4, cpu_count())
SEED = 42

OSD_ORDER = 1
OSD_SEARCH_BITS = 0
OSD_MAX_CANDIDATES = 4096

PROTOCOLS = [
    ShiftInvariantPR(
        name="shift-invariant-pr/osd",
        c=C,
        osd=OSDConfig(
            order=OSD_ORDER,
            search_bits=OSD_SEARCH_BITS,
            max_candidates=OSD_MAX_CANDIDATES,
        ),
    ),
]
# Add another encoder/decoder spec here; its name must be unique.
OVERHEAD_BINS = 24
SNR_BINS = 24

CONFIG = ExperimentConfig(
    experiments=EXPEREMENTS,
    message_sizes=K_VALUES,
    snr_db_range=SNR_DB_RANGE,
    overhead_range=OVERHEAD_RANGE,
    shift_range=S_RANGE,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

## Prepare shared fields, carriers and generator prefixes

One immutable bank per degree is shared across message lengths, protocol instances and CPU threads.
Only the requested channel window is encoded; lost symbols are not calculated on each trial.
The pilot is field element **1** (coordinate bits `1, 0, ...`), on the carrier after the data carriers.


In [3]:
PREPARED = prepare_protocols(PROTOCOLS, CONFIG)
layout_rows = []
for protocol, variants in PREPARED.items():
    for K, asset in variants.items():
        layout_rows.append(
            dict(
                Proto=protocol,
                K=K,
                candidates=choose_degree(K, C)[2],
                k=asset.degree,
                padding=asset.padding,
                D=asset.dimension,
                components=len(asset.exponents),
                primitive_carriers=len(asset.binary_field.carriers),
                modulus=hex(asset.binary_field.modulus),
                N_max=asset.max_length,
            )
        )
display(pd.DataFrame(layout_rows))

,Proto,K,candidates,k,padding,D,components,primitive_carriers,modulus,N_max
0,shift-invariant-pr/osd,32,"(9, 10, 11)",11,1,44,4,176,0x805,66
1,shift-invariant-pr/osd,64,"(10, 11, 12)",11,2,77,7,176,0x805,116
2,shift-invariant-pr/osd,128,"(11, 12, 13)",13,2,143,11,630,0x201b,215
3,shift-invariant-pr/osd,256,"(12, 13, 14)",13,4,273,21,630,0x201b,410


## Compile before measurement

This is a separate `njit` warm-up pass. It does not consume the experiment random stream.


In [4]:
warmup_protocols(PREPARED)
print("Encoder, OSD and phase-repair kernels are ready.")

Encoder, OSD and phase-repair kernels are ready.


## Run transmissions

Joblib schedules batches on CPU threads; Numba releases the GIL inside each numerical batch.
All workers share the same read-only precomputation. Each trial chooses one protocol independently.
Per-chunk seeds make results independent of worker count and completion order.

Every run computes all trials from scratch and keeps the table in notebook memory.
`succ` compares the recovered K message bits exactly. Neither padding checks nor
knowledge of S enters decoding.


In [5]:
trials = run_experiment(
    PROTOCOLS,
    CONFIG,
    prepared=PREPARED,
    workers=WORKERS,
)
print(f"{len(trials):,} trials in {trials.attrs['elapsed_seconds']:.1f} seconds")
display(trials[["Proto", "K", "Overhead", "SNR", "succ"]].head())
display(
    trials.groupby(["Proto", "K"], observed=True).agg(
        trials=("succ", "size"),
        success_rate=("succ", "mean"),
        valid_rate=("valid", "mean"),
    )
)

Protocol batches:   0%|          | 0/3907 [00:00<?, ?it/s]

1,000,000 trials in 57.4 seconds


,Proto,K,Overhead,SNR,succ
0,shift-invariant-pr/osd,32,0.141361,-0.741843,False
1,shift-invariant-pr/osd,256,0.335058,2.919789,True
2,shift-invariant-pr/osd,128,0.168237,1.306561,True
3,shift-invariant-pr/osd,64,0.470048,2.707168,True
4,shift-invariant-pr/osd,64,0.332424,-2.289129,False


trials  success_rate  valid_rate
Proto                  K                                    
shift-invariant-pr/osd 32   250258      0.511312    0.999744
                       64   250282      0.440339    0.999752
                       128  250310      0.350949    0.999920
                       256  249150      0.253076    0.999900

## Conditional success probability

Three views, each with a filter for the remaining variable:
- **K vs Overhead**: SNR interval or ALL.
- **SNR vs K**: Overhead interval or ALL.
- **Overhead vs SNR**: K or ALL.

The first named variable is initially on X. **Swap X/Y** transposes each chart independently.
K is discrete. SNR and Overhead filters select bins defined by SNR_BINS and OVERHEAD_BINS;
intervals include the lower edge, and only the last interval includes its upper edge.
**ALL** disables the slider and pools observations across the remaining variable.
Probabilities use pooled success/observation counts, not averages of bin probabilities.

Empty cells are blank. Hover shows bin population; mouse-wheel zoom is enabled.
All views use the existing `trials` table, without rerunning the experiment.


In [6]:
if "success_view" in globals():
    success_view.close()
for view in globals().get("success_views", []):
    view.close()
# Reload plotting dependencies so rerunning this cell picks up UI changes.
# The existing trials table and prepared decoder assets stay in memory.
import importlib
import _shared.application.protocol_experiment as protocol_experiment

importlib.reload(protocol_experiment)
import _shared.presentation.protocol_plots as protocol_plots

importlib.reload(protocol_plots)

success_views = []
for axes in (("K", "Overhead"), ("SNR", "K"), ("Overhead", "SNR")):
    view = protocol_plots.success(
        trials,
        axes=axes,
        overhead_range=OVERHEAD_RANGE,
        snr_db_range=SNR_DB_RANGE,
        bins=(OVERHEAD_BINS, SNR_BINS),
    )
    success_views.append(view)
    display(view)

ProtocolView(children=(FigureWidget({
    'data': [{'colorbar': {'len': 0.9, 'outlinewidth': 0, 'thickness': 1…

ProtocolView(children=(FigureWidget({
    'data': [{'colorbar': {'len': 0.9, 'outlinewidth': 0, 'thickness': 1…

ProtocolView(children=(FigureWidget({
    'data': [{'colorbar': {'len': 0.9, 'outlinewidth': 0, 'thickness': 1…